# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rupamsingh0907-beep/Flyintern/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

I am provisionally choosing **Lane 2: Refresh / Content Opportunity Scoring**. The question it answers is which pages a content reviewer should look at first for refresh, expansion, protection, pruning or monitoring. I chose it because it has a clear person who acts (a reviewer with limited time) and a clear action to take on each page, so the work ends in a decision and not just a model. Data helps here because a fixed rule applies one cutoff to every page and cannot say which pages have the strongest evidence. In the starter data, 54.2% of pages are currently trending down, so a "declining" flag alone would send a reviewer to more than half of the pool. A ranked queue with reason codes could help prioritize within that pool. This is decision support: it suggests where to review first, and does not prove that a refresh will recover a page. I can still change lanes until the end of Week 4.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("content_refresh_anonymized.csv")

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision:** which pages a content reviewer should open first, out of a pool of about 30,000 pages.

**Who acts, and what they do:** a content editor or SEO reviewer with limited time, assumed here to be able to review about 50 pages in a cycle. For each page at the top of the queue, they choose one action: refresh, expand, protect, prune or monitor.

**Unit of analysis:** one page (one row per `content_id`).

**Output:** a ranked review queue. Each page gets a score, a suggested action and reason codes that say why it ranked high, such as "stale but visible" or "declining with demand."

**Cost of a wrong recommendation:** there are two kinds of error.
- *False alarm:* a page is ranked high but did not need work, so the reviewer's limited time is spent on it.
- *Missed page:* a page that is really losing visibility is ranked low and never reviewed, so the loss continues unnoticed.

Since reviewer time is the scarce resource, I expect false alarms to hurt more in daily use, because they fill the small top of the queue. A missed page can matter more for a high-traffic page, so the queue should be checked by a person and not applied automatically.

**Why data helps, and why this is not just "train a model":** with 54.2% of pages currently trending down, a flag like "declining" cannot tell a reviewer where to start. A ranking can order pages by how strong the evidence is, and it has to beat a transparent rule-based baseline to be worth using.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(df.shape)
print(list(df.columns))

(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

The starter file has 30,000 rows and 44 columns, with one row per page. It includes the signals Lane 2 needs: visibility (`impressions_90d`), engagement (`sessions_90d`, `engagement_rate`), freshness (`content_age_days`, `days_since_last_update`), and recent movement (`trend_direction`, `trend_pct`, plus last-30-day vs. previous-30-day columns). That is enough to build a review queue without collecting anything new.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
filtered = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
filtered = filtered.drop_duplicates(subset="content_id")

print("rows before:", len(df))
print("rows after filters:", len(filtered))
print("distinct clients:", filtered["client_id"].nunique())

rows before: 30000
rows after filters: 30000
distinct clients: 32


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What this work can say**
- **Observed:** counts and shares measured in the starter data, such as the number of pages, the number of clients, and the share of pages currently trending down.
- **Directional:** patterns in a 30,000-row slice, which suggest where to look but may not hold across all clients or time periods.
- **Decision-support:** a ranked review queue could help a reviewer prioritize which pages to open first. It suggests where to look, and a person still decides what to do.

**What this work cannot say**
- **No causal claims.** This data cannot show that refreshing, expanding or pruning a page causes a recovery. A page that improves after an edit may have improved anyway.
- **No claims about the future from a proxy.** `trend_direction` comes from the current window, so "currently trending down" is not the same as "will decline." A drop can also come from consolidation, seasonality, search-result or AI click loss, or noise, and this data alone cannot tell them apart.
- **No claims about Google.** Nothing here explains or predicts how Google ranks pages, and no signal in this data should be read as a ranking factor.
- **No client-level conclusions.** The pool combines 32 clients, so a pattern could be driven by a few large ones.

**Provisional status:** this lane is a first choice. I can confirm or change it until the end of Week 4.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(filtered["trend_direction"].value_counts())
print()
print(filtered["trend_direction"].value_counts(normalize=True).round(3))

down_share = (filtered["trend_direction"] == "down").mean()
print("share trending down:", round(down_share, 3))

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

trend_direction
down      0.542
stable    0.199
up        0.146
new       0.075
flat      0.038
Name: proportion, dtype: float64
share trending down: 0.542


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking
code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.